# BUS-BRA · Phi-3.5 Vision and CheX-Phi3.5V SFT

Attach or extract BUS-BRA with `bus_data.csv` beside its `Images` directory. Set `DATA_ROOT` below if running outside Kaggle. The original 400-case manifest fixes image bytes, order, labels and prompt. This notebook runs the real-image test and five trials per checkpoint with the same solid white 512 × 512 RGB image.

Run the cells in order in a fresh Kaggle or Colab Python GPU session with internet, PyTorch and matching torchvision. Use a 16 GB or larger GPU (preferably BF16-capable) and allow about 17 GB of cache space for the two checkpoints, plus the dataset. The models run one at a time. Run one copy per output directory. Rerunning resumes from the CSVs in the same output directory. The last cell bundles the outputs for download.

The setup cell starts a private Python kernel for the experiment. The following `%%phi` cells share that kernel, Kaggle’s GPU runtime and the working directory. Run the setup once per session.

For a full rerun, leave `PREVIOUS_OUTPUT = None` in a fresh session. To resume this white-control protocol, attach its saved output directory and set `PREVIOUS_OUTPUT` below to the folder containing `run_settings.json`. Use the same GPU and software settings. In a fresh run CheX-Phi goes first. Set `RUN_MODELS = ["chex_phi35_sft"]` to evaluate only the medical model.

In [ ]:
import atexit
import os
import subprocess
import sys
import tempfile
from contextlib import ExitStack
from pathlib import Path
from IPython.core.magic import register_cell_magic
from jupyter_client import KernelManager

# Keep the Phi libraries separate from Kaggle's preinstalled Hub, tokenizer and Pillow.
# The worker inherits the GPU build of torch/torchvision and the scientific Python stack.
if "phi_runtime" in globals():
    phi_runtime.close()
phi_runtime = ExitStack()
atexit.register(phi_runtime.close)
packages = Path(phi_runtime.enter_context(tempfile.TemporaryDirectory(prefix="bus-phi-packages-")))
pins = ["transformers==4.46.3", "tokenizers==0.20.3", "huggingface-hub==0.36.0",
        "accelerate==1.1.1", "pillow==11.3.0", "sentencepiece==0.2.1", "safetensors>=0.4.1", "regex"]
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                "--no-deps", "--target", str(packages), *pins], check=True)
env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(filter(None, [str(packages), env.get("PYTHONPATH")]))
manager = KernelManager(kernel_name="python3")
manager.kernel_spec.argv = [sys.executable, "-m", "ipykernel_launcher", "-f", "{connection_file}"]
manager.start_kernel(env=env, cwd=str(Path.cwd()))
phi_runtime.callback(manager.shutdown_kernel, now=True)
client = manager.client()
client.start_channels()
phi_runtime.callback(client.stop_channels)
client.wait_for_ready(timeout=120)

@register_cell_magic
def phi(line, cell):
    """Execute one experiment cell and relay its output to this notebook."""
    try:
        reply = client.execute_interactive(cell, timeout=None, allow_stdin=False)["content"]
    except KeyboardInterrupt:
        manager.interrupt_kernel()
        raise
    if reply["status"] != "ok":
        raise RuntimeError(f"Phi kernel: {reply.get('ename', reply['status'])}: {reply.get('evalue', '')}")

In [ ]:
%%phi
import csv, gc, hashlib, io, json, math, os, random, re, shutil, time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import torch
from PIL import Image
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoProcessor, AutoModelForCausalLM, LogitsProcessor, LogitsProcessorList
from tqdm import tqdm

from importlib.metadata import version
print("Phi runtime:", {p: version(p) for p in
      ("transformers", "tokenizers", "huggingface-hub", "accelerate", "pillow", "torch", "torchvision")})

MODEL_SPECS = {
    "phi35_vision": ("microsoft/Phi-3.5-vision-instruct", "12b77fb40b63a2c73c68243d3f767aab688a1b2a", ""),
    "chex_phi35_sft": ("HaiZhiYan/CheX-Phi35V", "fc6ff2f91c9ed4a6ec9b81d9315a10352a6d7ad4",
                       "CheX-Phi-3.5-vision-instruct-SFT"),
}
RUN_MODELS = ["chex_phi35_sft", "phi35_vision"]
PREVIOUS_OUTPUT = None  # Optional path to the prior output folder containing run_settings.json.
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DTYPE = torch.bfloat16  # Matches the study. Use a GPU with native BF16 support when available.
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

INPUT_ROOT = Path("/kaggle/input")
DATA_ROOT = None
OUTPUT_ROOT = WORK / "phi_bus_bra_white"
FINAL_SIZE, MAX_NEW_TOKENS = 400, 32
log = print
BUS_BRA_PROMPT = 'You are a medical assistant. Examine this breast ultrasound image.\nClassify the lesion as exactly ONE of the following:\n- benign: non-cancerous lesion\n- malignant: cancerous lesion\n\nRespond with ONLY the label: benign or malignant.'
METADATA_HASHES = {'f218d0f10b0aeeb14ff4b2cfed8631b2f4a7cf608e564548b35bddd55f3aafb3': 'original', '1a7756e34fa27cda4faa48f41c1b5a5443e2bd0af8aa2a41e24ad8b1a96ede71': 'kaggle_orvile'}

## Frozen BUS-BRA cohort

In [ ]:
%%phi
FROZEN_IMAGE_IDS = """
bus_0001-l bus_0003-r bus_0007-l bus_0008-r bus_0009-r bus_0010-r bus_0011-l bus_0012-r
bus_0017-s bus_0029-s bus_0030-s bus_0034-r bus_0037-s bus_0041-l bus_0042-r bus_0043-s
bus_0045-r bus_0048-l bus_0049-r bus_0050-l bus_0052-r bus_0053-l bus_0054-s bus_0057-l
bus_0058-r bus_0059-l bus_0076-r bus_0082-s bus_0084-l bus_0088-s bus_0092-r bus_0095-s
bus_0098-l bus_0099-r bus_0100-l bus_0104-s bus_0105-r bus_0106-r bus_0109-s bus_0114-s
bus_0118-l bus_0120-l bus_0122-s bus_0124-r bus_0130-r bus_0133-l bus_0135-r bus_0136-l
bus_0141-r bus_0143-r bus_0145-r bus_0146-r bus_0147-r bus_0148-l bus_0149-r bus_0156-r
bus_0157-l bus_0163-r bus_0165-l bus_0166-r bus_0171-l bus_0172-r bus_0174-l bus_0179-l
bus_0185-l bus_0186-l bus_0187-l bus_0191-l bus_0193-r bus_0197-r bus_0199-l bus_0200-s
bus_0205-l bus_0206-l bus_0208-r bus_0212-r bus_0214-l bus_0217-r bus_0218-l bus_0219-l
bus_0225-r bus_0230-r bus_0231-r bus_0232-r bus_0233-s bus_0234-l bus_0236-l bus_0239-l
bus_0240-r bus_0242-r bus_0243-r bus_0247-r bus_0249-r bus_0250-r bus_0260-l bus_0262-l
bus_0263-l bus_0264-r bus_0265-s bus_0268-r bus_0269-r bus_0274-r bus_0277-l bus_0281-l
bus_0282-r bus_0284-r bus_0286-r bus_0287-l bus_0288-r bus_0289-l bus_0290-l bus_0291-r
bus_0293-s bus_0301-r bus_0303-r bus_0310-l bus_0311-r bus_0319-l bus_0320-l bus_0321-s
bus_0324-s bus_0326-r bus_0328-r bus_0329-l bus_0331-s bus_0333-s bus_0334-l bus_0335-l
bus_0336-s bus_0340-l bus_0342-r bus_0344-r bus_0345-l bus_0351-l bus_0353-l bus_0360-l
bus_0364-r bus_0366-l bus_0368-l bus_0370-r bus_0371-l bus_0373-r bus_0374-l bus_0375-r
bus_0376-l bus_0379-l bus_0381-l bus_0384-r bus_0385-l bus_0387-r bus_0389-r bus_0390-r
bus_0392-r bus_0394-r bus_0395-l bus_0396-l bus_0397-l bus_0398-r bus_0399-r bus_0400-r
bus_0405-l bus_0410-l bus_0411-l bus_0413-l bus_0414-r bus_0415-l bus_0416-l bus_0418-r
bus_0421-r bus_0422-r bus_0426-l bus_0427-l bus_0431-r bus_0432-l bus_0438-r bus_0440-r
bus_0442-r bus_0445-r bus_0446-r bus_0448-l bus_0449-r bus_0453-l bus_0454-r bus_0455-l
bus_0456-l bus_0457-l bus_0458-l bus_0464-r bus_0465-l bus_0466-l bus_0471-l bus_0472-l
bus_0475-l bus_0484-r bus_0486-r bus_0488-l bus_0493-r bus_0497-l bus_0502-r bus_0507-l
bus_0508-l bus_0509-r bus_0513-r bus_0518-l bus_0521-l bus_0522-l bus_0531-r bus_0532-r
bus_0543-l bus_0544-l bus_0546-l bus_0549-l bus_0550-l bus_0551-r bus_0555-l bus_0560-r
bus_0566-l bus_0568-r bus_0569-l bus_0571-l bus_0574-r bus_0576-r bus_0579-l bus_0581-r
bus_0582-l bus_0590-l bus_0594-r bus_0595-r bus_0596-l bus_0600-r bus_0601-r bus_0602-l
bus_0603-r bus_0604-l bus_0605-l bus_0609-l bus_0612-r bus_0620-l bus_0626-r bus_0628-l
bus_0630-l bus_0631-l bus_0633-l bus_0639-r bus_0641-l bus_0644-r bus_0645-r bus_0647-l
bus_0657-r bus_0659-r bus_0661-r bus_0662-l bus_0663-r bus_0666-r bus_0667-r bus_0669-l
bus_0672-l bus_0677-s bus_0683-s bus_0685-l bus_0687-s bus_0697-r bus_0700-s bus_0705-s
bus_0706-s bus_0707-s bus_0709-s bus_0711-s bus_0712-s bus_0713-r bus_0721-s bus_0723-s
bus_0726-s bus_0728-s bus_0730-s bus_0731-s bus_0742-s bus_0745-s bus_0747-s bus_0748-s
bus_0752-s bus_0758-s bus_0760-s bus_0765-s bus_0766-s bus_0767-s bus_0768-s bus_0770-s
bus_0772-s bus_0791-s bus_0793-s bus_0797-s bus_0801-s bus_0803-s bus_0807-s bus_0809-s
bus_0811-s bus_0812-s bus_0816-s bus_0818-s bus_0820-s bus_0825-s bus_0826-s bus_0829-s
bus_0831-s bus_0836-s bus_0838-s bus_0841-s bus_0842-s bus_0846-s bus_0850-l bus_0854-s
bus_0858-l bus_0859-r bus_0860-l bus_0861-l bus_0862-r bus_0866-r bus_0867-r bus_0868-r
bus_0869-r bus_0874-l bus_0879-r bus_0881-l bus_0882-r bus_0886-s bus_0893-l bus_0896-r
bus_0899-s bus_0906-l bus_0911-l bus_0915-l bus_0917-s bus_0918-l bus_0927-l bus_0928-r
bus_0933-l bus_0934-l bus_0935-r bus_0937-l bus_0938-l bus_0941-r bus_0943-s bus_0944-r
bus_0945-r bus_0947-l bus_0948-r bus_0951-r bus_0952-l bus_0954-l bus_0957-s bus_0960-l
bus_0962-l bus_0964-r bus_0966-l bus_0971-r bus_0972-s bus_0973-l bus_0977-l bus_0979-r
bus_0980-r bus_0981-l bus_0982-l bus_0983-l bus_0985-l bus_0986-r bus_0988-s bus_0989-l
bus_0991-s bus_0993-r bus_0994-r bus_0997-l bus_0998-s bus_1002-l bus_1003-r bus_1007-l
bus_1008-r bus_1012-l bus_1013-r bus_1014-l bus_1016-l bus_1017-l bus_1023-r bus_1024-r
bus_1027-l bus_1028-r bus_1032-l bus_1033-l bus_1035-l bus_1037-r bus_1038-l bus_1042-l
bus_1046-l bus_1052-l bus_1055-l bus_1056-r bus_1058-l bus_1060-s bus_1061-r bus_1063-r
""".split()
PILOT_CASES = ['1004', '1049', '184', '196', '228', '253', '257', '31', '314', '369', '383', '412', '450', '462', '577', '610', '800', '901', '909', '961']
EXCLUDED_CASES = ['15', '178', '300', '735', '77', '78', '856', '857', '912', '931', '970', '976', '990']
EXPECTED_MANIFEST_SHA256 = 'a8caa6ad39310c2871e53bc4b9f80d4b76c1b6f850b1c474c0abae8db1181283'

In [ ]:
%%phi
from PIL import Image

search_root = Path(DATA_ROOT) if DATA_ROOT else INPUT_ROOT
metadata_files = sorted(search_root.rglob("bus_data.csv"))
if len(metadata_files) != 1:
    raise ValueError("Attach an extracted BUS-BRA dataset with Images and bus_data.csv; set DATA_ROOT if ambiguous")
SOURCE_ROOT = metadata_files[0].parent
metadata_sha = hashlib.sha256(metadata_files[0].read_bytes()).hexdigest()
if metadata_sha not in METADATA_HASHES:
    raise ValueError("BUS-BRA metadata does not match either verified dataset copy")
with metadata_files[0].open(newline="", encoding="utf-8-sig") as stream:
    metadata = list(csv.DictReader(stream))
by_id = {r["ID"]: r for r in metadata}
source_index = {r["ID"]: i for i, r in enumerate(metadata)}
if len(metadata) != 1875 or len(by_id) != len(metadata):
    raise ValueError("Unexpected metadata count or duplicate image IDs")
image_dirs = [p for p in SOURCE_ROOT.iterdir() if p.is_dir() and p.name.lower() == "images"]
if len(image_dirs) != 1:
    raise ValueError("Expected one Images folder beside bus_data.csv")
image_paths = defaultdict(list)
for path in image_dirs[0].rglob("*"):
    if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}:
        image_paths[path.stem].append(path)

log("Loading the frozen final sample and verifying image content")
SAMPLE = []
for image_id in FROZEN_IMAGE_IDS:
    row = by_id[image_id]
    if row["Case"] in set(PILOT_CASES) | set(EXCLUDED_CASES):
        raise ValueError("Final sample overlaps pilot or excluded Cases")
    paths = image_paths[image_id]
    if len(paths) != 1:
        raise ValueError(f"Missing or ambiguous image: {image_id}")
    path = paths[0]
    SAMPLE.append({"dataset": "bus_bra", "sample_id": image_id, "image_id": image_id,
                   "image_path": path.relative_to(SOURCE_ROOT).as_posix(), "group_id": row["Case"],
                   "split": "test", "task_type": "classification", "question": BUS_BRA_PROMPT,
                   "options_json": "{}", "reference_answer": row["Pathology"].strip().lower(),
                   "category": "", "image_sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
if (len(SAMPLE) != FINAL_SIZE or len({r["sample_id"] for r in SAMPLE}) != FINAL_SIZE
        or len({r["group_id"] for r in SAMPLE}) != FINAL_SIZE
        or Counter(r["reference_answer"] for r in SAMPLE) != {"benign": 200, "malignant": 200}):
    raise ValueError("Expected 400 distinct images/Cases with 200 images per class")
# Match the original frozen manifest's exact CSV serialization without exporting another file.
manifest = io.StringIO(newline="")
writer = csv.DictWriter(manifest, fieldnames=list(SAMPLE[0]))
writer.writeheader()
writer.writerows(SAMPLE)
if hashlib.sha256(manifest.getvalue().encode()).hexdigest() != EXPECTED_MANIFEST_SHA256:
    raise ValueError("Frozen sample mismatch: image content, labels, Cases, paths or prompt changed")
log("Sample ready: 400 images / 400 source Cases; 200 benign + 200 malignant")
log("Dataset:", METADATA_HASHES[metadata_sha], "| manifest SHA256:", EXPECTED_MANIFEST_SHA256)
log("Fixed prompt:", repr(BUS_BRA_PROMPT))

## Classification prompt and answer parsing

In [ ]:
%%phi
def parse_answer(text):
    """Strict, deterministic final-answer parser; no label search inside reasoning."""
    response = text.strip()
    if not response:
        return None, "empty"
    # Exactly one balanced reasoning block may precede a final answer.
    if response.startswith("<think>"):
        if response.count("<think>") != 1 or response.count("</think>") != 1:
            return None, "incomplete_or_multiple_think_blocks"
        response = response.split("</think>", 1)[1].strip()
    if "<think>" in response or "</think>" in response:
        return None, "misplaced_think_tag"
    if "<answer>" in response or "</answer>" in response:
        match = re.fullmatch(r"<answer>\s*(benign|malignant)\s*</answer>", response, re.IGNORECASE)
        if match:
            return match.group(1).lower(), "answer_tag"
        return None, "malformed_or_ambiguous_answer_tag"
    match = re.fullmatch(r"(?:final answer:\s*)?(benign|malignant)", response, re.IGNORECASE)
    if match:
        return match.group(1).lower(), "direct_or_final_marker"
    return None, "no_unambiguous_final_answer"

def prediction(item, raw):
    label, _ = parse_answer(raw)
    return label or "unparsed"

ITEMS = [
    dict(idx=source_index[r["sample_id"]], question_id=r["sample_id"], group="lesion_classification",
         anatomy="breast", question=BUS_BRA_PROMPT, true_answer=r["reference_answer"],
         prompt=BUS_BRA_PROMPT, path=r["image_path"])
    for r in SAMPLE
]

def item_image(item, condition):
    with Image.open(SOURCE_ROOT / item["path"]) as source:
        return source.convert("RGB")

## Load a pinned checkpoint and generate a direct answer

In [ ]:
%%phi
class NumericsGuard(LogitsProcessor):
    def __init__(self):
        self.invalid = False

    def __call__(self, input_ids, scores):
        self.invalid |= bool((torch.isnan(scores) | torch.isposinf(scores)).any()
                             or torch.isneginf(scores).all(dim=-1).any())
        return scores


def load_chex_weights(checkpoint, config, device):
    from accelerate import init_empty_weights, load_checkpoint_and_dispatch
    from transformers import GenerationConfig

    # Accelerate 1.1.1 reads the pinned PyTorch shards even without safetensors format metadata.
    with init_empty_weights(include_buffers=False):
        model = AutoModelForCausalLM.from_config(
            config, trust_remote_code=True, torch_dtype=DTYPE, attn_implementation="eager",
        )
    model.tie_weights()
    model = load_checkpoint_and_dispatch(
        model, str(checkpoint / "model.safetensors.index.json"),
        device_map={"": device}, dtype=DTYPE, strict=True,
    )
    model.generation_config = GenerationConfig.from_pretrained(checkpoint)
    return model.eval()


def load_model(tag):
    if not torch.cuda.is_available():
        raise RuntimeError("Select a CUDA GPU runtime before starting inference.")
    repo, revision, subfolder = MODEL_SPECS[tag]
    prefix = subfolder + "/" if subfolder else ""
    files = ["*.py", "*.json", "*.safetensors", "*.model"]
    snapshot = snapshot_download(repo, revision=revision, token=HF_TOKEN,
                                 allow_patterns=[prefix + pattern for pattern in files])
    checkpoint = Path(snapshot) / subfolder
    processor = AutoProcessor.from_pretrained(checkpoint, trust_remote_code=True, num_crops=16)
    # Load the saved config first so the eager override takes precedence over its backend.
    config = AutoConfig.from_pretrained(checkpoint, trust_remote_code=True)
    if tag == "chex_phi35_sft":
        model = load_chex_weights(checkpoint, config, "cuda:0")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            checkpoint, config=config, trust_remote_code=True, torch_dtype=DTYPE, device_map={"": 0},
            attn_implementation="eager", use_safetensors=True,
        ).eval()
    if model.config._attn_implementation != "eager":
        raise RuntimeError(f"Expected eager attention, got {model.config._attn_implementation}")
    print(tag, revision, subfolder, torch.cuda.get_device_name(0), DTYPE, "attention=eager")
    return model, processor


@torch.inference_mode()
def generate(model, processor, image, prompt):
    tokenizer = processor.tokenizer
    messages = [{"role": "user", "content": "<|image_1|>\n" + prompt}]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(rendered, [image], return_tensors="pt")
    device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(device=device, dtype=DTYPE) if v.is_floating_point() else v.to(device)
              for k, v in inputs.items()}
    guard = NumericsGuard()
    # Phi's chat template ends turns with <|end|>. Stop there or at the tokenizer's EOS.
    end_ids = list(dict.fromkeys([tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|end|>")]))
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                         num_beams=1, use_cache=True, eos_token_id=end_ids,
                         pad_token_id=tokenizer.pad_token_id, logits_processor=LogitsProcessorList([guard]))
    tokens = out[0, inputs["input_ids"].shape[1]:].tolist()
    raw = tokenizer.decode(tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return raw, len(tokens), guard.invalid

## Save predictions and score the paired cohort

In [ ]:
%%phi
SAMPLE_COLUMNS = ["idx", "question_id", "group"]
RESULT_COLUMNS = [*SAMPLE_COLUMNS, "anatomy", "question", "true_answer", "predicted_answer", "raw_output"]
SCORE_COLUMNS = ["group", "correct", "n", "accuracy"]

def write_csv(path, records, fields):
    temp = path.with_suffix(".tmp")
    with temp.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(records)
    temp.replace(path)

def scores(records):
    groups = sorted({r["group"] for r in records})
    output = []
    for group in groups:
        answers = [r for r in records if r["group"] == group]
        correct = sum(r["true_answer"] == r["predicted_answer"] for r in answers)
        output.append(dict(group=group, correct=correct, n=len(answers), accuracy=correct / len(answers)))
    return output

def run_items(tag, condition, folder, prefix, model, processor):
    folder.mkdir(parents=True, exist_ok=True)
    write_csv(folder / f"{prefix}_sample.csv", ITEMS, SAMPLE_COLUMNS)
    path = folder / f"{prefix}_results_{tag}.csv"
    rows = []
    if path.exists():
        with path.open(encoding="utf-8", newline="") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != RESULT_COLUMNS:
                raise ValueError(f"Unexpected columns in {path}")
            rows = list(reader)
        keys = RESULT_COLUMNS[:-2]
        if len(rows) > len(ITEMS) or any(
            any(row[k] != str(item[k]) for k in keys) for row, item in zip(rows, ITEMS)
        ):
            raise ValueError(f"Saved results do not match the frozen sample: {path}")
    for item in tqdm(ITEMS[len(rows):], desc=f"{tag} {condition}"):
        image = item_image(item, condition)
        raw, _, invalid = generate(model, processor, image, item["prompt"])
        pred = "unparsed" if invalid else prediction(item, raw)
        rows.append({**item, "predicted_answer": pred, "raw_output": raw})
        write_csv(path, rows, RESULT_COLUMNS)
        if len(rows) == 1 or len(rows) % 25 == 0:
            print(f"{tag} {condition}: saved {len(rows)}/{len(ITEMS)} predictions", flush=True)
    summary = scores(rows)
    write_csv(folder / f"{prefix}_scores_{tag}.csv", summary, SCORE_COLUMNS)
    print(tag, condition, sum(r["correct"] for r in summary), "/", len(rows),
          "correct;", sum(r["predicted_answer"] == "unparsed" for r in rows), "unparsed")
    display(pd.DataFrame(summary))
    return rows

In [ ]:
%%phi
if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime before starting the run.")
settings = {
    "protocol": 2, "conditions": ["real", "white"], "dataset_revision": EXPECTED_MANIFEST_SHA256, "models": MODEL_SPECS, "seed": SEED, "dtype": str(DTYPE),
    "max_new_tokens": MAX_NEW_TOKENS, "num_crops": 16, "attention": "eager",
    "torch": str(torch.__version__), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "sample_and_prompts_sha256": hashlib.sha256(json.dumps(ITEMS, sort_keys=True).encode()).hexdigest(),
}
settings_text = json.dumps(settings, indent=2, sort_keys=True)
if PREVIOUS_OUTPUT and not OUTPUT_ROOT.exists():
    from tempfile import TemporaryDirectory
    previous = Path(PREVIOUS_OUTPUT)
    if (previous / "run_settings.json").read_text() != settings_text:
        raise ValueError(f"Previous run settings differ: {previous}")
    with TemporaryDirectory(dir=OUTPUT_ROOT.parent) as staging:
        restored = Path(staging) / "results"
        shutil.copytree(previous, restored)
        restored.rename(OUTPUT_ROOT)
    print(f"Restored previous predictions from {previous}", flush=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
record = OUTPUT_ROOT / "run_settings.json"
if record.exists() and record.read_text() != settings_text:
    raise ValueError("Run settings changed. Select a new output directory for this run.")
record.write_text(settings_text)

## White-image stability trials

Each model generates five responses to the white image. The notebook saves them in `control_trials.csv` and derives 400 matched predictions only when all five raw and parsed responses agree. On restart the notebook restores individual completed trials before deriving the matched outputs.

In [ ]:
%%phi
TRIAL_COLUMNS = ["model", "model_revision", "color", "trial", "parsed_answer",
                 "parse_status", "parse_reason", "raw_output", "generated_tokens",
                 "latency_ms", "visual_input_pixel_sha256", "completed_at", "status"]
SUMMARY_COLUMNS = ["model", "color", "completed_trials", "expected_trials", "stable",
                   "parsed_prediction", "raw_output", "derived_rows", "derived_accuracy",
                   "unique_raw_outputs", "unique_parsed_predictions", "model_revision", "status"]


def run_controls(tag, model, processor):
    revision = MODEL_SPECS[tag][1]
    control_root = OUTPUT_ROOT / "blank_controls" / tag
    control_root.mkdir(parents=True, exist_ok=True)
    trials_path = control_root / "control_trials.csv"
    trials = []
    if trials_path.exists():
        with trials_path.open(encoding="utf-8", newline="") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != TRIAL_COLUMNS:
                raise ValueError(f"Unexpected trial columns in {trials_path}")
            trials = list(reader)
    color = "white"
    expected_order = [(color, number) for number in range(1, 6)]
    if ([(t["color"], int(t["trial"])) for t in trials] != expected_order[:len(trials)]
            or any(t["model"] != tag or t["model_revision"] != revision or t["status"] != "ok"
                   for t in trials)):
        raise ValueError(f"Saved control trials do not match this experiment: {trials_path}")
    folder = control_root / color
    folder.mkdir(exist_ok=True)
    image = Image.new("RGB", (512, 512), color)
    digest = hashlib.sha256(image.tobytes()).hexdigest()
    for repeat in range(len(trials) + 1, 6):
        start = time.perf_counter()
        raw, length, invalid = generate(model, processor, image, BUS_BRA_PROMPT)
        label, reason = parse_answer(raw)
        if invalid:
            label, reason = None, "invalid_scores"
        trial = dict(model=tag, model_revision=revision, color=color, trial=repeat,
                     parsed_answer=label or "", parse_status="parsed" if label else "unparsed",
                     parse_reason=reason, raw_output=raw, generated_tokens=length,
                     latency_ms=1000 * (time.perf_counter() - start),
                     visual_input_pixel_sha256=digest, completed_at=datetime.now(timezone.utc).isoformat(),
                     status="ok")
        trials.append(trial)
        write_csv(trials_path, trials, TRIAL_COLUMNS)
    distinct = {(t["raw_output"], t["parsed_answer"], t["parse_reason"]) for t in trials}
    stable = len(distinct) == 1
    first = trials[0]
    label = first["parsed_answer"] or "unparsed"
    derived = [{**item, "predicted_answer": label, "raw_output": first["raw_output"]}
               for item in ITEMS] if stable else []
    summary = scores(derived) if stable else [
        dict(group="lesion_classification", correct=0, n=0, accuracy=None)]
    write_csv(folder / "vqa_sample.csv", ITEMS, SAMPLE_COLUMNS)
    write_csv(folder / f"vqa_results_{tag}.csv", derived, RESULT_COLUMNS)
    write_csv(folder / f"vqa_scores_{tag}.csv", summary, SCORE_COLUMNS)
    summaries = [dict(model=tag, color=color, completed_trials=5, expected_trials=5,
                          stable=stable, parsed_prediction=label if stable else "unstable",
                          raw_output=first["raw_output"] if stable else "", derived_rows=len(derived),
                          derived_accuracy=summary[0]["accuracy"],
                          unique_raw_outputs=len({t["raw_output"] for t in trials}),
                          unique_parsed_predictions=len({t["parsed_answer"] for t in trials}),
                          model_revision=revision, status="ready_for_paired_comparison" if stable
                          else "unstable_or_incomplete")]
    write_csv(control_root / "control_summary.csv", summaries, SUMMARY_COLUMNS)
    display(pd.DataFrame(summaries))

In [ ]:
%%phi
for tag in RUN_MODELS:
    model, processor = load_model(tag)
    try:
        run_items(tag, "real", OUTPUT_ROOT / "real", "classification", model, processor)
        run_controls(tag, model, processor)
    finally:
        del model, processor
        gc.collect()
        torch.cuda.empty_cache()

## Paired real-image versus blank-image comparisons

Align by the frozen case ID, count changes in correctness, and report the class-specific results, exact McNemar test and paired case-bootstrap intervals for each model with a stable white control.

In [ ]:
%%phi
import numpy as np
EXPECTED_N = FINAL_SIZE
EXPECTED_CLASS_COUNTS = {"benign": 200, "malignant": 200}
BOOTSTRAP_REPS, BOOTSTRAP_SEED = 10_000, SEED

REQUIRED_COLUMNS = {'question_id', 'true_answer', 'predicted_answer', 'question', 'raw_output'}
CLASSES = {'benign', 'malignant'}
ALLOWED_PREDICTIONS = CLASSES | {'unparsed'}


def load_result(path, condition):
    if not path.is_file():
        raise FileNotFoundError(
            f'{condition}: Missing {path}. Check your directory configuration and the downloaded result files.'
        )
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f'{condition}: Missing expected columns: {sorted(missing)}')
    if len(df) != EXPECTED_N:
        raise ValueError(f'{condition}: Expected {EXPECTED_N} rows, got {len(df)}. '
                         'Finish the inference before comparing.')
    if df.question_id.eq('').any() or df.question_id.duplicated().any():
        raise ValueError(f'{condition}: Missing/duplicate question_id values.')
    for column in ('true_answer', 'predicted_answer'):
        df[column] = df[column].str.strip().str.lower()
    if not set(df.true_answer).issubset(CLASSES):
        raise ValueError(f'{condition}: Invalid ground-truth labels: {set(df.true_answer) - CLASSES}')
    if df.true_answer.value_counts().to_dict() != EXPECTED_CLASS_COUNTS:
        raise ValueError(f'{condition}: Wrong reference-label distribution: '
                         f'{df.true_answer.value_counts().to_dict()}')
    invalid_predictions = set(df.predicted_answer) - ALLOWED_PREDICTIONS
    if invalid_predictions:
        raise ValueError(f'{condition}: Invalid predictions: {invalid_predictions}')
    return df


def pair_results(model_name, tag, color):
    filename = f'vqa_results_{tag}.csv'
    real = load_result(OUTPUT_ROOT / 'real' / f'classification_results_{tag}.csv', f'{model_name} real')
    blank = load_result(OUTPUT_ROOT / 'blank_controls' / tag / color / filename, f'{model_name} blank')
    real_ids, blank_ids = set(real.question_id), set(blank.question_id)
    if real_ids != blank_ids:
        raise ValueError(f'{model_name}: Different sample IDs. '
                         f'Missing from blank={sorted(real_ids - blank_ids)[:5]}; '
                         f'extra in blank={sorted(blank_ids - real_ids)[:5]}')

    merged = real.merge(blank, on='question_id', how='inner', suffixes=('_real', '_blank'),
                        validate='one_to_one')
    for col in ('true_answer', 'question'):
        left, right = f'{col}_real', f'{col}_blank'
        if not merged[left].equals(merged[right]):
            raise ValueError(f'{model_name}: Mismatched {col} for the same question_id.')
    if 'group_real' in merged and 'group_blank' in merged:
        if not merged['group_real'].equals(merged['group_blank']):
            raise ValueError(f'{model_name}: Mismatched groups for the same question_id.')

    merged = merged.rename(columns={
        'true_answer_real': 'true_answer',
        'question_real': 'question',
        'predicted_answer_real': 'real_prediction',
        'predicted_answer_blank': 'blank_prediction',
        'raw_output_real': 'real_raw_output',
        'raw_output_blank': 'blank_raw_output',
    })
    merged = merged.drop(columns=['true_answer_blank', 'question_blank'])
    merged['real_correct'] = merged.real_prediction.eq(merged.true_answer)
    merged['blank_correct'] = merged.blank_prediction.eq(merged.true_answer)
    merged['prediction_changed'] = merged.real_prediction.ne(merged.blank_prediction)
    conditions = [
        merged.real_correct & ~merged.blank_correct,
        ~merged.real_correct & merged.blank_correct,
        merged.real_correct & merged.blank_correct,
    ]
    choices = ['real_only_correct', 'blank_only_correct', 'both_correct']
    merged['paired_outcome'] = np.select(conditions, choices, default='both_wrong')
    merged.insert(0, 'model', model_name)
    return merged.sort_values('question_id').reset_index(drop=True)

from scipy.stats import binomtest


def summarize_paired(df, *, bootstrap_reps=BOOTSTRAP_REPS, seed=BOOTSTRAP_SEED):
    n = len(df)
    counts = df.paired_outcome.value_counts()
    real_only = int(counts.get('real_only_correct', 0))
    blank_only = int(counts.get('blank_only_correct', 0))
    both_correct = int(counts.get('both_correct', 0))
    both_wrong = int(counts.get('both_wrong', 0))
    discordant = real_only + blank_only
    p_value = (float(binomtest(min(real_only, blank_only), discordant, p=0.5).pvalue)
               if discordant else 1.0)

    # Each item is one matched Case: +100 if real correct / blank wrong,
    # -100 for the opposite, and 0 when correctness is tied.
    differences = (df.real_correct.astype(int) - df.blank_correct.astype(int)).to_numpy()
    rng = np.random.default_rng(seed)
    boot_means = np.empty(bootstrap_reps, dtype=float)
    for b in range(bootstrap_reps):
        boot_means[b] = differences[rng.integers(0, n, n)].mean() * 100
    low, high = np.quantile(boot_means, [0.025, 0.975])

    return {
        'model': df.model.iat[0],
        'n': n,
        'real_correct': int(df.real_correct.sum()),
        'blank_correct': int(df.blank_correct.sum()),
        'real_accuracy_pct': round(float(df.real_correct.mean() * 100), 3),
        'blank_accuracy_pct': round(float(df.blank_correct.mean() * 100), 3),
        'real_minus_blank_pp': round(float(differences.mean() * 100), 3),
        'paired_bootstrap_95ci_low_pp': round(float(low), 3),
        'paired_bootstrap_95ci_high_pp': round(float(high), 3),
        'real_only_correct': real_only,
        'blank_only_correct': blank_only,
        'both_correct': both_correct,
        'both_wrong': both_wrong,
        'changed_predictions': int(df.prediction_changed.sum()),
        'changed_predictions_pct': round(float(df.prediction_changed.mean() * 100), 3),
        'mcnemar_exact_two_sided_p': p_value,
        'blank_predicts_benign': int(df.blank_prediction.eq('benign').sum()),
        'blank_predicts_malignant': int(df.blank_prediction.eq('malignant').sum()),
        'blank_unparsed': int(df.blank_prediction.eq('unparsed').sum()),
        'real_unparsed': int(df.real_prediction.eq('unparsed').sum()),
    }


color = "white"
PAIRED = {}
for tag in RUN_MODELS:
    trial_summary = pd.read_csv(OUTPUT_ROOT / "blank_controls" / tag / "control_summary.csv")
    if trial_summary.loc[trial_summary.color == color, "derived_rows"].item() != FINAL_SIZE:
        print(f"Skipping paired analysis for {tag}/{color}: unstable control")
        continue
    PAIRED[tag] = pair_results(tag, tag, color)
if PAIRED:
    output = OUTPUT_ROOT / "paired" / color
    output.mkdir(parents=True, exist_ok=True)
    summary = pd.DataFrame([summarize_paired(df) for df in PAIRED.values()])
    display(summary)

    per_class_rows = []
    for name, df in PAIRED.items():
        for label, group in df.groupby('true_answer'):
            per_class_rows.append({
                'model': name,
                'true_answer': label,
                'n': len(group),
                'real_accuracy_pct': round(group.real_correct.mean() * 100, 3),
                'blank_accuracy_pct': round(group.blank_correct.mean() * 100, 3),
                'real_only_correct': int((group.real_correct & ~group.blank_correct).sum()),
                'blank_only_correct': int((~group.real_correct & group.blank_correct).sum()),
            })
    per_class = pd.DataFrame(per_class_rows)
    display(per_class)

    summary.to_csv(output / "paired_model_summary.csv", index=False)
    per_class.to_csv(output / "paired_per_class_summary.csv", index=False)
    pd.concat(PAIRED.values(), ignore_index=True).to_csv(output / "paired_predictions_all_models.csv", index=False)
    for tag, paired in PAIRED.items():
        paired.to_csv(output / f"paired_predictions_{tag}.csv", index=False)

In [ ]:
%%phi
archive = shutil.make_archive(str(OUTPUT_ROOT), "zip", OUTPUT_ROOT)
print("Result bundle:", archive)
from IPython.display import FileLink, display
display(FileLink(os.path.relpath(archive)))